# Домашнее задание по теме "Механизм внимания"

## Задание

1. Возьмите англо-русскую пару фраз ([www.manythings.org....org/anki/](https://www.manythings.org/anki/))
1. Обучите на них seq2seq with attention
    - На основе скалярного произведения
    - На основе MLP
1. Оцените качество

In [25]:
from io import open
import unicodedata
import string
import re
import random
import gc

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepare Data

In [4]:
!tail ../datas/data/eng-fra.txt

Five tremors in excess of magnitude 5.0 on the Richter scale have shaken Japan just this week, but scientists are warning that the largest expected aftershock has yet to hit.	Cinq secousses dépassant la magnitude cinq sur l'échelle de Richter ont secoué le Japon précisément cette semaine, mais les scientifiques avertissent que la plus grande réplique est encore à venir.
No matter how much you try to convince people that chocolate is vanilla, it'll still be chocolate, even though you may manage to convince yourself and a few others that it's vanilla.	Peu importe le temps que tu passeras à essayer de convaincre les gens que le chocolat est de la vanille, ça restera toujours du chocolat, même si tu réussis à convaincre toi et quelques autres que c'est de la vanille.
A child who is a native speaker usually knows many things about his or her language that a non-native speaker who has been studying for years still does not know and perhaps will never know.	Un enfant qui est un locuteur natif

In [5]:
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [6]:
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    # Добавляем пробел перед знаками препинания
    s = re.sub(r"([.!?])", r" \1", s)
    # Убираем всё, кроме латиницы, кириллицы и знаков препинания
    s = re.sub(r"[^a-zA-Zа-яА-Я.!?]+", r" ", s)
    return s.strip()

In [7]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Если файл называется rus.txt, используем просто имя. 
    # Если хочешь динамически: '../datas/%s-%s.txt' % (lang1, lang2)
    lines = open('../datas/rus.txt', encoding='utf-8').read().strip().split('\n')

    # ИСПРАВЛЕНО: Берем только первые две колонки [:2], игнорируя метаданные
    pairs = [[normalizeString(s) for s in l.split('\t')[:2]] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [9]:
MAX_LENGTH = 10

# Эти префиксы важны для фильтрации датасета, чтобы модель училась на простых фразах
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    # p[0] - входной язык, p[1] - выходной (английский, если reverse=True)
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [10]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

# ВАЖНО: Если файл rus.txt, то lang2 должен быть 'rus'
input_lang, output_lang, pairs = prepareData('eng', 'rus', True)
print(random.choice(pairs))

Reading lines...
Read 536124 sentence pairs
Trimmed to 30628 sentence pairs
Counting words...
Counted words:
rus 10313
eng 4289
['я уверен что том может победить .', 'i m sure that tom can win .']


## Позиционное внимание (Location-based Attention)

### The Encoder

In [39]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

### The Decoder

In [40]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        attn_weights = F.softmax(
            self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)

        output = F.relu(output)
        output, hidden = self.gru(output, hidden)

        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [29]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [30]:
teacher_forcing_ratio = 0.5


def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [31]:
import time
import math


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [41]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.001):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [33]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [34]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

In [35]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, attentions = evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [36]:
def cleanup_gpu():
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory cleared. Current allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [42]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder1 = AttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

trainIters(encoder1, attn_decoder1, 150000, print_every=10000)

1m 19s (- 18m 27s) (10000 6%) 2.5525
2m 34s (- 16m 44s) (20000 13%) 1.9797
3m 50s (- 15m 21s) (30000 20%) 1.8100
5m 6s (- 14m 2s) (40000 26%) 1.6950
6m 22s (- 12m 45s) (50000 33%) 1.6372
7m 38s (- 11m 27s) (60000 40%) 1.5789
8m 54s (- 10m 10s) (70000 46%) 1.5133
10m 9s (- 8m 53s) (80000 53%) 1.4950
11m 26s (- 7m 37s) (90000 60%) 1.4547
12m 43s (- 6m 21s) (100000 66%) 1.4235
13m 59s (- 5m 5s) (110000 73%) 1.4127
15m 16s (- 3m 49s) (120000 80%) 1.3829
16m 33s (- 2m 32s) (130000 86%) 1.3679
17m 50s (- 1m 16s) (140000 93%) 1.3159
19m 6s (- 0m 0s) (150000 100%) 1.3213


In [43]:
evaluateRandomly(encoder1, attn_decoder1)

> мы больше не встречаемся .
= we re not seeing each other anymore .
< we re not longer anymore anymore . <EOS>

> я вернусь сегодня ночью .
= i m coming back tonight .
< i m going to tonight . <EOS>

> я уверена что он рано уидет .
= i m sure he ll leave early .
< i m sure she ll ll leave early . <EOS>

> я привык к запаху .
= i m used to the smell .
< i m used to the dark . <EOS>

> я мама тома .
= i m tom s mom .
< i m tom s mother . <EOS>

> ты ведь несчастен да ?
= you re unhappy aren t you ?
< you re unhappy aren t you ? <EOS>

> я не уверена что мы можем доверять тому .
= i m not sure we can trust tom .
< i m not sure we can trust . . <EOS>

> ты обманщик .
= you re a fraud .
< you re a liar . <EOS>

> они просто пытаются помочь .
= they re only trying to help .
< they re just trying to help . <EOS>

> я рассчитываю на то что вы будете сильнои .
= i m counting on you to be strong .
< i m counting on what you do . <EOS>



In [45]:
cleanup_gpu()

GPU memory cleared. Current allocated: 277.43 MB


## Скалярное произведение (Dot-Product Attention)

### The Decoder

In [ ]:
class DotAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DotAttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size * 2, output_size) # Вход: GRU_out + context

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        gru_output, hidden = self.gru(embedded, hidden)
        # Скалярное произведение)
        attn_scores = torch.bmm(gru_output, encoder_outputs.unsqueeze(0).transpose(1, 2))
        attn_weights = F.softmax(attn_scores, dim=2)
        # Применяем веса к выходам энкодера (получаем контекст)
        context = torch.bmm(attn_weights, encoder_outputs.unsqueeze(0))
        combined = torch.cat((gru_output.squeeze(0), context.squeeze(0)), 1)
        output = F.log_softmax(self.out(combined), dim=1)

        return output, hidden, attn_weights

In [49]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
dot_decoder1 = DotAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

trainIters(encoder1, dot_decoder1, 150000, print_every=10000)

1m 13s (- 17m 5s) (10000 6%) 2.4783
2m 23s (- 15m 35s) (20000 13%) 1.8908
3m 33s (- 14m 15s) (30000 20%) 1.7107
4m 43s (- 13m 0s) (40000 26%) 1.6126
5m 54s (- 11m 48s) (50000 33%) 1.5488
7m 4s (- 10m 36s) (60000 40%) 1.4687
8m 14s (- 9m 25s) (70000 46%) 1.4196
9m 24s (- 8m 14s) (80000 53%) 1.3973
10m 34s (- 7m 3s) (90000 60%) 1.3350
11m 45s (- 5m 52s) (100000 66%) 1.2908
12m 55s (- 4m 41s) (110000 73%) 1.2644
14m 5s (- 3m 31s) (120000 80%) 1.2625
15m 16s (- 2m 20s) (130000 86%) 1.2195
16m 26s (- 1m 10s) (140000 93%) 1.2096
17m 36s (- 0m 0s) (150000 100%) 1.1678


In [50]:
evaluateRandomly(encoder1, dot_decoder1)

> мне нужны только вы .
= you re the only one i need .
< i m the only one you . <EOS>

> мы ждем вашего ответа .
= we re waiting for your answer .
< we re waiting for your answer . <EOS>

> у меня правыи глаз не видит .
= i m blind in the right eye .
< i m blind in the right eye . <EOS>

> я уверен что том этого не сделает .
= i m sure tom won t do that .
< i m sure that tom won t do that .

> я все тебе расскажу .
= i m going to tell you everything .
< i m going to tell everything tell everything . <EOS>

> мы с нетерпением ждем возможности увидеть вас снова .
= we re looking forward to seeing you again .
< i m looking forward to again again . <EOS>

> рад это слышать .
= i m glad to hear it .
< i m glad that hear hear hear . <EOS>

> ты худшая ученица в классе .
= you re the worst student in the class .
< you re the the the student . <EOS>

> я слишком устала чтобы спорить .
= i m too tired to argue .
< i m not tired to talk . <EOS>

> рада с вами познакомиться .
= i m happy to meet 

In [51]:
cleanup_gpu()

GPU memory cleared. Current allocated: 299.41 MB


## MLP Attention

### The Decoder

In [54]:
class MLPAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(MLPAttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        # MLP слои для вычисления весов
        self.attn_mlp = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1) # Схлопываем в одно число (score)
        
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size * 2, output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        gru_output, hidden = self.gru(embedded, hidden)
        max_len = encoder_outputs.size(0)
        h_repeated = gru_output.repeat(max_len, 1, 1).transpose(0, 1) 
        e_outputs = encoder_outputs.unsqueeze(0)
        # MLP: tanh(W * [h; e])
        combined_energy = torch.tanh(self.attn_mlp(torch.cat((h_repeated, e_outputs), 2)))
        attn_scores = self.v(combined_energy).transpose(1, 2) # (1, 1, max_len)
        attn_weights = F.softmax(attn_scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.unsqueeze(0))
        combined = torch.cat((gru_output.squeeze(0), context.squeeze(0)), 1)
        output = F.log_softmax(self.out(combined), dim=1)

        return output, hidden, attn_weights

In [55]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
mlp_decoder1 = MLPAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

trainIters(encoder1, mlp_decoder1, 150000, print_every=10000)

1m 20s (- 18m 42s) (10000 6%) 2.3743
2m 37s (- 17m 2s) (20000 13%) 1.7203
3m 54s (- 15m 37s) (30000 20%) 1.5098
5m 11s (- 14m 16s) (40000 26%) 1.3908
6m 28s (- 12m 57s) (50000 33%) 1.3507
7m 46s (- 11m 39s) (60000 40%) 1.2638
9m 3s (- 10m 21s) (70000 46%) 1.2360
10m 21s (- 9m 3s) (80000 53%) 1.2040
11m 37s (- 7m 45s) (90000 60%) 1.1633
12m 54s (- 6m 27s) (100000 66%) 1.1258
14m 11s (- 5m 9s) (110000 73%) 1.1069
15m 27s (- 3m 51s) (120000 80%) 1.0885
16m 44s (- 2m 34s) (130000 86%) 1.0720
18m 0s (- 1m 17s) (140000 93%) 1.0413
19m 17s (- 0m 0s) (150000 100%) 1.0355


In [56]:
evaluateRandomly(encoder1, mlp_decoder1)

> я куплю рыбу .
= i m going to buy fish .
< i m going to buy wine fish . <EOS>

> мне очень жарко .
= i m very hot .
< i m very hot . <EOS>

> мы печем торты .
= we re baking cakes .
< we re baking too . <EOS>

> они не такие .
= they re different .
< they re not like . <EOS>

> ты уважаемыи мнои человек .
= you re a person i respect .
< you re a person person person . <EOS>

> она гордится своеи дочерью .
= she is proud of her daughter .
< she is proud of her daughter . <EOS>

> я рад что идет дождь .
= i m glad it s raining .
< i m glad it s raining . <EOS>

> я удивлен что том нас помнит .
= i m surprised tom remembers us .
< i m surprised tom remembers remembers us . <EOS>

> мне просто скучно .
= i m just bored .
< i m just bored . <EOS>

> вы гораздо быстрее меня .
= you re much faster than i am .
< you re much faster than i am . <EOS>



## Оценка качества моделей

1. **MLP Attention** показал лучший результат (Loss **1.03**), что эффективнее Dot-Product и лучше позиционного внимания.

2. В отличие от первой модели **Location-based**, которая ориентировалась на «номер слова в строке», **Dot-Product** и **MLP** перешли к контентному анализу. Это позволило модели подбирать синонимы (*mother* вместо *mom*, *liar* вместо *fraud*), понимая **суть** слова, а не его место в предложении.

3. **Скорость обучения и ресурсы:**
    - **Dot-Product** — самый быстрый и легкий механизм (всего 17 мин),
    - **MLP** — самый тяжелый и долгий (19 мин), так как видеокарте приходится обучать дополнительную мини-нейросеть внутри слоя внимания.

5. Все три модели на базе RNN сохранили склонность к повторам и редким «галлюцинациям», хотя **MLP** максимально приблизило результат к идеальному.